In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv('C:/Users/MihajloTesic/Desktop/Python fajlovi/IMDB-Dataset.csv')

In [8]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000))
])

param_grid = {
    'tfidf__max_features': [3000, 5000, 10000],
    'tfidf__ngram_range': [(1,1),(1,2)],
    'tfidf__min_df': [1, 5],
    'tfidf__max_df': [0.8, 0.9],
    'clf__C': [0.01, 0.1, 1, 10],
    'clf__penalty': ['l2']
}

grid = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=2)

random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    n_jobs=-1,
    verbose=2
)

In [4]:
X = df['review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [5]:
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [9]:
random_search.fit(X_train, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


,estimator,Pipeline(step..._iter=1000))])
,param_distributions,"{'clf__C': [0.01, 0.1, ...], 'clf__penalty': ['l2'], 'tfidf__max_df': [0.8, 0.9], 'tfidf__max_features': [3000, 5000, ...], ...}"
,n_iter,10
,scoring,None
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


# L2 (Ridge) regularization

In [34]:
penalty1 = 'l2'
C = 0.01

In [35]:
model1 = LogisticRegression(penalty=penalty1, C=C)

model1.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,0.01
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [36]:
y_pred1 = model1.predict(X_test_tfidf)

In [37]:
print('Accuracy: ', accuracy_score(y_test, y_pred))
print('\nClassification Report:\n')
print(classification_report(y_test, y_pred))

Accuracy:  0.826

Classification Report:

              precision    recall  f1-score   support

    negative       0.85      0.79      0.82      7411
    positive       0.81      0.86      0.83      7589

    accuracy                           0.83     15000
   macro avg       0.83      0.83      0.83     15000
weighted avg       0.83      0.83      0.83     15000



# L1 (Lasso) regularization

In [38]:
penalty2 = 'l1'
C = 0.01

In [39]:
model2 = LogisticRegression(penalty=penalty2, C=C)

model2.fit(X_train_tfidf, y_train)

ValueError: Solver lbfgs supports only 'l2' or None penalties, got l1 penalty.

AttributeError: 'GridSearchCV' object has no attribute 'best_estimator_'

In [11]:
def predict_sentiment(text, model):
    text_tfidf = vectorizer.transform([text])
    prediction = model.predict(text_tfidf)
    return prediction[0]

def predict_sentiment_randomizedCV(text):
    return random_search.predict([text])[0]

In [47]:
print(predict_sentiment('This was a good film.'))
print(predict_sentiment('This was a horrible film.'))
print(predict_sentiment('The film was so good, that I had to turn it off.'))
print(predict_sentiment('The film was good, although there were some horrible scenes.'))
print(predict_sentiment('A bad ending, but I loved the film.'))

positive
negative
positive
negative
negative


In [12]:
print(predict_sentiment_randomizedCV('This was a good film.'))
print(predict_sentiment_randomizedCV('This was a horrible film.'))
print(predict_sentiment_randomizedCV('The film was so good, that I had to turn it off.'))
print(predict_sentiment_randomizedCV('The film was good, although there were some horrible scenes.'))
print(predict_sentiment_randomizedCV('A bad ending, but I loved the film.'))

positive
negative
positive
negative
positive
